In [ ]:
import pandas as pd
import re

In [13]:
# ========= 1. 文件路径 =========
file_path = "/Users/littlestars/Desktop/agricultural_data/TAB4.1_4.10-сигнал.xls"   # 改成你的实际文件名
output_path = "grain_panel_merged.csv"
# ========= 2. 读所有sheet =========
xls = pd.ExcelFile(file_path)
sheet_names = [s for s in xls.sheet_names if str(s).startswith("4.2")]
print("发现这些 4.2 工作表：")
for s in sheet_names:
    print(" -", s)
all_long = []

# ========= 3. 一些辅助函数 =========
def normalize_region(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)

    # 去掉明显无效行
    bad_patterns = [
        r"^Российская Федерация",
        r"федеральный округ",
        r"^в том числе",
        r"без автономных округов",
        r"автономн",
        r"^Продолжение табл",
        r"^СУБЪЕКТЫ РОССИЙСКОЙ ФЕДЕРАЦИИ",
        r"^4\.2",
        r"^тыс\.? ?тонн",
        r"^$"
    ]
    for p in bad_patterns:
        if re.search(p, x, flags=re.IGNORECASE):
            return None

    # 这类“城市行”通常不纳入农业地区面板，可先剔除
    if x.startswith("г."):
        return None

    return x


def to_number(x):
    try:
        if pd.isna(x):
            return None
    except:
        return None

    x = str(x).strip()

    if x in ["-", "—", "…", "..."]:
        return None

    if "р." in x:
        return None

    x = x.replace(",", ".")

    try:
        return float(x)
    except:
        return None


# ========= 4. 逐个sheet处理 =========
for sheet in sheet_names:
    print(f"\n正在处理: {sheet}")

    df = pd.read_excel(file_path, sheet_name=sheet, header=None)

    # 找到包含年份的表头行
    year_row_idx = None
    years = {}

    for i in range(min(30, len(df))):
        row = df.iloc[i]
        for j, val in enumerate(row):
            if pd.notna(val):
                text = str(val).strip()
                if re.fullmatch(r"(19|20)\d{2}", text):
                    year_row_idx = i
                    years[j] = int(text)

        if len(years) >= 2:
            break

    if year_row_idx is None or len(years) == 0:
        print(f"  跳过 {sheet}：没找到年份行")
        continue

    print("  识别到年份列：", years)

    # 数据从年份行下方开始
    data = df.iloc[year_row_idx + 1:].copy()

    # A列默认是地区
    region_col = 0

    keep_cols = [region_col] + sorted(years.keys())
    data = data.iloc[:, keep_cols].copy()

    new_cols = ["region"] + [str(years[c]) for c in sorted(years.keys())]
    data.columns = new_cols

    # 地区清洗
    data["region"] = data["region"].apply(normalize_region)
    data = data.dropna(subset=["region"])

    # 数值清洗
    year_cols = [c for c in data.columns if c != "region"]
    for c in year_cols:
        data[c] = data[c].apply(to_number)

    # 全部年份都空的行删掉
    data = data.dropna(how="all", subset=year_cols)

    # 转成长表
    long_df = data.melt(
        id_vars="region",
        var_name="year",
        value_name="production"
    )

    long_df = long_df.dropna(subset=["production"])
    long_df["year"] = long_df["year"].astype(int)

    # 你的单位是“千吨”，这里统一转成“吨”
    long_df["production"] = long_df["production"] * 1000

    all_long.append(long_df)

# ========= 5. 合并全部sheet =========
final_df = pd.concat(all_long, ignore_index=True)

# 去重
final_df = final_df.drop_duplicates(subset=["region", "year"], keep="first")

# 排序
final_df = final_df.sort_values(["region", "year"]).reset_index(drop=True)

# 保存
final_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\n合并完成。")
print(final_df.head(20))
print(f"\n已保存到: {output_path}")

发现这些 4.2 工作表：
 - 4.2 (16)
 - 4.2 (15)
 - 4.2 (14)
 - 4.2 (13)
 - 4.2 (12)
 - 4.2 (11)
 - 4.2 (10)
 - 4.2 (9)
 - 4.2 (8)
 - 4.2 (7)
 - 4.2 (6)
 - 4.2 (5)
 - 4.2 (4)
 - 4.2 (3)
 - 4.2 (2)
 - 4.2

正在处理: 4.2 (16)
  识别到年份列： {5: 2005, 3: 2005}

正在处理: 4.2 (15)
  识别到年份列： {2: 2005, 3: 2006, 6: 2005}

正在处理: 4.2 (14)
  识别到年份列： {2: 2006, 3: 2007, 6: 2006}

正在处理: 4.2 (13)
  识别到年份列： {1: 2006, 2: 2007}

正在处理: 4.2 (12)
  识别到年份列： {1: 2007, 2: 2008}

正在处理: 4.2 (11)
  识别到年份列： {1: 2008, 2: 2009}

正在处理: 4.2 (10)
  识别到年份列： {1: 2009, 2: 2010}

正在处理: 4.2 (9)
  识别到年份列： {1: 2010, 2: 2011}

正在处理: 4.2 (8)


/var/folders/49/jcjz_zr14yz2f5w13_38hgh80000gn/T/ipykernel_29091/4112119024.py:119: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  data[c] = data[c].apply(to_number)
/var/folders/49/jcjz_zr14yz2f5w13_38hgh80000gn/T/ipykernel_29091/4112119024.py:119: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  data[c] = data[c].apply(to_number)
/var/folders/49/jcjz_zr14yz2f5w13_38hgh80000gn/T/ipykernel_29091/4112119024.py:119: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position

  识别到年份列： {1: 2011, 2: 2012}

正在处理: 4.2 (7)
  识别到年份列： {1: 2012, 2: 2013}

正在处理: 4.2 (6)
  识别到年份列： {1: 2013, 2: 2014, 3: 2015}

正在处理: 4.2 (5)
  识别到年份列： {1: 2014, 2: 2015, 3: 2016}

正在处理: 4.2 (4)
  识别到年份列： {1: 2015, 2: 2016, 3: 2017}

正在处理: 4.2 (3)
  识别到年份列： {1: 2017, 2: 2018, 3: 2019}

正在处理: 4.2 (2)
  识别到年份列： {1: 2019, 2: 2020, 3: 2021}

正在处理: 4.2
  识别到年份列： {1: 2021, 2: 2022, 3: 2023}

合并完成。
              region  year production
0     Алтайский край  2006  3518000.0
1     Алтайский край  2007  4698310.0
2     Алтайский край  2008  3857490.0
3     Алтайский край  2009  5627845.0
4     Алтайский край  2010  4240800.0
5     Алтайский край  2011  3919500.0
6     Алтайский край  2012  2516800.0
7     Алтайский край  2013  4926100.0
8     Алтайский край  2014  3294900.0
9     Алтайский край  2015  3940400.0
10    Алтайский край  2016  4829700.0
11    Алтайский край  2017  4975500.0
12    Алтайский край  2018  5010700.0
13    Алтайский край  2019  4591700.0
14    Алтайский край  2020  3951100.

In [15]:
df = pd.read_csv("grain_panel_merged.csv")

summary = df.groupby("region")["year"].agg(["min", "max", "nunique"]).reset_index()
print(summary.sort_values(["min", "max"]).head(20))
print(summary.sort_values("nunique").head(20))

                             region   min   max  nunique
17               Камчатская область  2006  2006        1
74                Читинская область  2006  2007        2
2             Архангельская область  2006  2016       11
48                  Республика Коми  2006  2016       11
3              Астраханская область  2006  2021       16
0                    Алтайский край  2006  2023       18
1                  Амурская область  2006  2023       18
4              Белгородская область  2006  2023       18
5                  Брянская область  2006  2023       18
6              Владимирская область  2006  2023       18
7             Волгоградская область  2006  2023       18
8               Вологодская область  2006  2023       18
9               Воронежская область  2006  2023       18
10               Забайкальский край  2006  2023       18
11               Ивановская область  2006  2023       18
12                Иркутская область  2006  2023       18
13              Кoстромская обл

In [21]:
# ========= 1. 删除时间太短的地区 =========
count = df.groupby("region")["year"].nunique()
valid_regions = count[count >= 10].index  # 至少10年

df = df[df["region"].isin(valid_regions)].copy()

# ========= 2. 清理地区名称 =========
def clean_region(x):
    x = str(x)

    # 去掉脚注
    x = re.sub(r"\*\)", "", x)

    # 修复奇怪字符（拉丁o → 俄文o）
    x = x.replace("Кoстромская", "Костромская")

    # 去空格
    x = x.strip()

    return x

df["region"] = df["region"].apply(clean_region)

# ========= 3. 删除明显垃圾行 =========
df = df[~df["region"].str.contains("округ", na=False)]
df = df[~df["region"].str.contains("Российская Федерация", na=False)]

# ========= 4. 删除不完整地区（可选但建议） =========
bad_regions = [
    "Камчатская область",
    "Читинская область"
]

df = df[~df["region"].isin(bad_regions)]

# ========= 5. 最终排序 =========
df = df.sort_values(["region", "year"]).reset_index(drop=True)

# ========= 6. 保存 =========
df.to_csv("grain_panel_final.csv", index=False, encoding="utf-8-sig")

print("清洗完成")
print(df.head())

清洗完成
           region  year  production
0  Алтайский край  2006   3518000.0
1  Алтайский край  2007   4698310.0
2  Алтайский край  2008   3857490.0
3  Алтайский край  2009   5627845.0
4  Алтайский край  2010   4240800.0


In [23]:
check = df[df["year"].isin([2005, 2006])].sort_values(["region", "year"])
print(check.head(100))

dup = df.groupby(["region", "year"]).size().reset_index(name="n")
print(dup[dup["n"] > 1].head(50))

                     region  year  production
0            Алтайский край  2006   3518000.0
18         Амурская область  2006    233690.0
36    Архангельская область  2006      3700.0
47     Астраханская область  2006     38980.0
63     Белгородская область  2006   1645900.0
...                     ...   ...         ...
1208       Хабаровский край  2006     17860.0
1226    Челябинская область  2006   2114650.0
1244   Чеченская Республика  2006    129040.0
1262   Чувашская Республика  2006    426030.0
1280    Ярославская область  2006     81260.0

[73 rows x 3 columns]
Empty DataFrame
Columns: [region, year, n]
Index: []


In [27]:
df.groupby("year").size()

year
2006    73
2007    73
2008    73
2009    73
2010    73
2011    73
2012    73
2013    73
2014    74
2015    73
2016    73
2017    71
2018    71
2019    70
2020    70
2021    71
2022    70
2023    71
dtype: int64

In [1]:
import xarray as xr

ds = xr.open_dataset("/Users/littlestars/Desktop/climate_variables.nc")

print(ds)

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


<xarray.Dataset>
Dimensions:     (valid_time: 216, latitude: 411, longitude: 1601)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2006-01-01 2006-02-01 ... 2023-12-01
  * latitude    (latitude) float64 82.0 81.9 81.8 81.7 ... 41.3 41.2 41.1 41.0
  * longitude   (longitude) float64 20.0 20.1 20.2 20.3 ... 179.8 179.9 180.0
Data variables:
    number      int64 ...
    expver      (valid_time) object ...
    t2m         (valid_time, latitude, longitude) float32 ...
    tp          (valid_time, latitude, longitude) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-20T15:30 GRIB to CDM+CF via cfgrib-0.9.1...


In [3]:
import xarray as xr
import pandas as pd

# 读文件
ds = xr.open_dataset("/Users/littlestars/Desktop/climate_variables.nc")

# 转成表
df = ds[["t2m", "tp"]].to_dataframe().reset_index()

# 单位转换
df["t2m"] = df["t2m"] - 273.15   # 开尔文 -> 摄氏度
df["tp"] = df["tp"] * 1000       # 米 -> 毫米

# 提取年份
df["year"] = pd.to_datetime(df["valid_time"]).dt.year

# 对整个俄罗斯区域做年度聚合
climate_yearly = df.groupby("year").agg(
    temp=("t2m", "mean"),
    rain=("tp", "sum")
).reset_index()

print(climate_yearly)
climate_yearly.to_csv("/Users/littlestars/Desktop/climate_yearly.csv", index=False, encoding="utf-8-sig")

    year      temp        rain
0   2006 -1.908756  7239684.50
1   2007 -0.491959  7451264.00
2   2008 -0.843856  7355052.00
3   2009 -1.935137  6995376.50
4   2010 -2.015379  7153761.00
5   2011 -1.072380  6894304.50
6   2012 -1.424770  7140261.00
7   2013 -1.218792  7337171.50
8   2014 -1.443777  7092941.00
9   2015 -0.696523  7325095.00
10  2016 -0.783957  7326448.00
11  2017 -0.707801  7366581.50
12  2018 -1.131990  7061673.50
13  2019 -0.527446  7289428.50
14  2020  0.513925  7293646.00
15  2021 -1.290350  7320562.00
16  2022 -0.537780  6166387.50
17  2023 -0.518525  3924516.25


In [5]:
import pandas as pd

grain = pd.read_csv("/Users/littlestars/Desktop/grain_panel_final.csv")
climate = pd.read_csv("/Users/littlestars/Desktop/climate_yearly.csv")

final_df = grain.merge(climate, on="year", how="left")

print(final_df.head())
print(final_df.shape)

final_df.to_csv("/Users/littlestars/Desktop/final_dataset.csv", index=False, encoding="utf-8-sig")

           region  year  production      temp       rain
0  Алтайский край  2006   3518000.0 -1.908756  7239684.5
1  Алтайский край  2007   4698310.0 -0.491959  7451264.0
2  Алтайский край  2008   3857490.0 -0.843856  7355052.0
3  Алтайский край  2009   5627845.0 -1.935137  6995376.5
4  Алтайский край  2010   4240800.0 -2.015379  7153761.0
(1298, 5)


In [25]:
df = pd.read_csv("/Users/littlestars/Desktop/final_dataset.csv")

regions = (
    df["region"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print(regions)
print("地区数量：", len(regions))

regions.to_csv("/Users/littlestars/Desktop/region_list.csv", index=False, encoding="utf-8-sig")

0            Алтайский край
1          Амурская область
2     Архангельская область
3      Астраханская область
4      Белгородская область
              ...          
69         Хабаровский край
70      Челябинская область
71     Чеченская Республика
72     Чувашская Республика
73      Ярославская область
Name: region, Length: 74, dtype: str
地区数量： 74


In [27]:
region_centroids = pd.DataFrame({
    "region": regions,
    "lat": None,
    "lon": None
})

region_centroids.to_csv(
    "/Users/littlestars/Desktop/region_centroids_template.csv",
    index=False,
    encoding="utf-8-sig"
)

print(region_centroids.head())

                  region   lat   lon
0         Алтайский край  None  None
1       Амурская область  None  None
2  Архангельская область  None  None
3   Астраханская область  None  None
4   Белгородская область  None  None


In [1]:
import pandas as pd
import time
from geopy.geocoders import Nominatim

# 读模板
centroids = pd.read_csv("/Users/littlestars/Desktop/region_centroids_template.csv")

geolocator = Nominatim(user_agent="russia_region_locator")

lats = []
lons = []

for region in centroids["region"]:
    query = f"{region}, Russia"
    try:
        location = geolocator.geocode(query, timeout=10)
        if location is not None:
            lats.append(location.latitude)
            lons.append(location.longitude)
            print(region, "->", location.latitude, location.longitude)
        else:
            lats.append(None)
            lons.append(None)
            print(region, "-> 未找到")
    except Exception as e:
        lats.append(None)
        lons.append(None)
        print(region, "-> 报错:", e)

    time.sleep(1)  # 别太快，避免被限制

centroids["lat"] = lats
centroids["lon"] = lons

centroids.to_csv(
    "/Users/littlestars/Desktop/region_centroids_auto.csv",
    index=False,
    encoding="utf-8-sig"
)

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Алтайский край -> 52.6932243 82.6931424
Амурская область -> 52.8032368 128.437295
Архангельская область -> 63.5589686 43.1221646
Астраханская область -> 47.1878186 47.608851
Белгородская область -> 50.7080119 37.5837615
Брянская область -> 52.8873315 33.415853
Владимирская область -> 56.0503336 40.6561633
Волгоградская область -> 49.6048339 44.2903582
Вологодская область -> 60.0391461 43.1215213
Воронежская область -> 50.9800393 40.1506507
Забайкальский край -> 52.248521 115.956325
Ивановская область -> 56.9167446 41.4352137
Иркутская область -> 56.6370122 104.719221
Кабардино-Балкарская Республика -> 43.4428286 43.4204809
Калининградская область -> 54.7293041 21.1489473
Калужская область -> 54.4382773 35.5272854
Камчатский край -> 57.1914882 160.0383819
Карачаево-Черкесская Республика -> 43.7368326 41.7267991
Кемеровская область -> 54.5335781 87.342861
Кировская область -> 58.563072 50.1824997
Костромская область -> 58.424756 44.2533273
Краснодарский край -> 45.7684014 39.0261044
Крас

In [3]:
import xarray as xr
import pandas as pd

ds = xr.open_dataset("/Users/littlestars/Desktop/climate_variables.nc")
centroids = pd.read_csv("/Users/littlestars/Desktop/region_centroids_auto.csv")

out = []

for _, r in centroids.iterrows():
    if pd.isna(r["lat"]) or pd.isna(r["lon"]):
        continue

    sub = ds.sel(latitude=r["lat"], longitude=r["lon"], method="nearest")
    tmp = sub[["t2m", "tp"]].to_dataframe().reset_index()

    tmp["temp"] = tmp["t2m"] - 273.15
    tmp["rain"] = tmp["tp"] * 1000
    tmp["year"] = pd.to_datetime(tmp["valid_time"]).dt.year
    tmp["region"] = r["region"]

    yearly = tmp.groupby(["region", "year"]).agg(
        temp=("temp", "mean"),
        rain=("rain", "sum")
    ).reset_index()

    out.append(yearly)

climate_region = pd.concat(out, ignore_index=True)

climate_region.to_csv(
    "/Users/littlestars/Desktop/climate_region_yearly.csv",
    index=False,
    encoding="utf-8-sig"
)

print(climate_region.head())

           region  year      temp       rain
0  Алтайский край  2006  3.104762  17.850813
1  Алтайский край  2007  4.821818  17.677519
2  Алтайский край  2008  3.944815  15.802220
3  Алтайский край  2009  2.169966  16.615139
4  Алтайский край  2010  1.055743  13.789200


In [5]:
grain = pd.read_csv("/Users/littlestars/Desktop/grain_panel_final.csv")
climate = pd.read_csv("/Users/littlestars/Desktop/climate_region_yearly.csv")

final_region_df = grain.merge(climate, on=["region", "year"], how="left")

print(final_region_df.head())
print(final_region_df.shape)

final_region_df.to_csv(
    "/Users/littlestars/Desktop/final_dataset_region_climate.csv",
    index=False,
    encoding="utf-8-sig"
)

           region  year  production      temp       rain
0  Алтайский край  2006   3518000.0  3.104762  17.850813
1  Алтайский край  2007   4698310.0  4.821818  17.677519
2  Алтайский край  2008   3857490.0  3.944815  15.802220
3  Алтайский край  2009   5627845.0  2.169967  16.615139
4  Алтайский край  2010   4240800.0  1.055743  13.789200
(1298, 5)


In [9]:
from linearmodels.panel import PanelOLS
import pandas as pd
df = pd.read_csv("final_dataset_region_climate.csv")
df = df.set_index(["region", "year"])

model = PanelOLS.from_formula(
    "production ~ temp + rain + EntityEffects + TimeEffects",
    data=df
).fit()

print(model)

                          PanelOLS Estimation Summary                           
Dep. Variable:             production   R-squared:                        0.0210
Estimator:                   PanelOLS   R-squared (Between):             -0.1117
No. Observations:                1298   R-squared (Within):              -0.0711
Date:                Sun, Mar 22 2026   R-squared (Overall):             -0.1071
Time:                        23:21:19   Log-likelihood                 -1.92e+04
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      12.939
Entities:                          74   P-value                           0.0000
Avg Obs:                       17.541   Distribution:                  F(2,1205)
Min Obs:                      10.0000                                           
Max Obs:                       18.000   F-statistic (robust):             12.939
                            

In [13]:
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv("final_dataset_region_climate.csv")

# 数值处理
df["production"] = pd.to_numeric(df["production"], errors="coerce")
df["temp"] = pd.to_numeric(df["temp"], errors="coerce")
df["rain"] = pd.to_numeric(df["rain"], errors="coerce")

df = df.dropna().copy()

# 固定效应
region_dummies = pd.get_dummies(df["region"].astype(str), drop_first=True, dtype=float)
year_dummies = pd.get_dummies(df["year"].astype(str), drop_first=True, dtype=float)

X = pd.concat([df[["temp", "rain"]], region_dummies, year_dummies], axis=1)
X = sm.add_constant(X)

y = df["production"].astype(float)

# 关键：用稳健标准误
model = sm.OLS(y, X).fit(cov_type="HC1")

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             production   R-squared:                       0.922
Model:                            OLS   Adj. R-squared:                  0.916
Method:                 Least Squares   F-statistic:                     97.03
Date:                Fri, 20 Mar 2026   Prob (F-statistic):               0.00
Time:                        21:18:23   Log-Likelihood:                -19200.
No. Observations:                1298   AIC:                         3.859e+04
Df Residuals:                    1205   BIC:                         3.907e+04
Df Model:                          92                                         
Covariance Type:                  HC1                                         
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 